<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/claude%2Fkind-volta-wlb7e5/vllm_tpu/vllm_tpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vLLM on a Colab TPU (v5e-1)

Runtime → Change runtime type → **v5e-1 TPU**.

**Read this first.** One v5e chip has **16 GB of HBM**. `Qwen/Qwen3.8-27B` is about 27.8B parameters, roughly **55 GB in bf16**, so it cannot fit.
vLLM's TPU backend loads safetensors weights (bf16 or supported quantizations), not the GGUF Q4 file used in the llama.cpp notebook. There is no official FP8/AWQ/GPTQ build of this model to shrink it either.
This notebook therefore checks whether the model fits, and if not, **falls back to a smaller Qwen model** so you still get a working vLLM-on-TPU run. Whether the 27B architecture is even supported by the TPU backend is unverified.

## 1. Check the TPU and host

In [ ]:
!free -h | head -2
!df -h /content | tail -1
import os; print({k: v for k, v in os.environ.items() if "TPU" in k or "PJRT" in k})

## 2. Config

In [ ]:
TARGET_MODEL    = "Qwen/Qwen3.8-27B"   # what you asked for
TARGET_PARAMS_B = 27.8                  # billions of parameters
FALLBACK_MODEL  = "Qwen/Qwen3-4B"      # ~8 GB in bf16; fits one v5e chip
FALLBACK_PARAMS_B = 4.0

HBM_GB          = 16
BYTES_PER_PARAM = 2          # bf16
KV_AND_OVERHEAD_GB = 3       # rough headroom for KV cache and activations
MAX_MODEL_LEN   = 4096
FORCE_TARGET    = False      # True = try the 27B anyway (expect an out-of-memory failure)

## 3. Does it fit?

In [ ]:
def need_gb(params_b): return params_b * BYTES_PER_PARAM + KV_AND_OVERHEAD_GB

print(f"{TARGET_MODEL}: needs about {need_gb(TARGET_PARAMS_B):.0f} GB, chip has {HBM_GB} GB")
if need_gb(TARGET_PARAMS_B) <= HBM_GB or FORCE_TARGET:
    MODEL_ID = TARGET_MODEL
else:
    MODEL_ID = FALLBACK_MODEL
    print(f"Does not fit. Falling back to {FALLBACK_MODEL} (about {need_gb(FALLBACK_PARAMS_B):.0f} GB).")
print("Using:", MODEL_ID)

## 4. Install vLLM with the TPU backend

In [ ]:
%%capture install_log
# Package names and versions for the TPU backend change often; see https://github.com/vllm-project/tpu-inference
!pip install -q vllm-tpu
# If the line above fails, try: !pip install -q tpu-inference vllm

In [ ]:
import importlib.metadata as md_
for pkg in ("vllm", "vllm-tpu", "tpu-inference", "jax", "jaxlib"):
    try: print(pkg, md_.version(pkg))
    except md_.PackageNotFoundError: print(pkg, "not installed")
import jax; print(jax.devices())

## 5. Start the server (the first start compiles with XLA and can take several minutes)

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(
    ["vllm", "serve", MODEL_ID,
     "--max-model-len", str(MAX_MODEL_LEN),
     "--tensor-parallel-size", "1",
     "--host", "127.0.0.1", "--port", "8000"],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(600):                      # up to ~20 min
    if server.poll() is not None:
        print("server exited:\n", open("/content/vllm.log").read()[-4000:]); break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/vllm.log").read()[-4000:])

## 6. Chat (OpenAI-compatible API)

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions", json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": temperature,
    }, timeout=900)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

print(chat("Explain what a TPU is in two sentences."))

## Notes
- Check `/content/vllm.log` for errors. Unsupported architectures, out-of-memory, and version mismatches show up there.
- Host RAM on the Colab TPU runtime is limited (about 12 GB reported), which also constrains loading large checkpoints.
- Colab disk is wiped when the session ends, so models re-download each session.